In [1]:
from f5_tts.model.cfm import CFM

from f5_tts.model.backbones.unett import UNetT
from f5_tts.model.backbones.dit import DiT
from f5_tts.model.backbones.mmdit import MMDiT

from f5_tts.model.trainer import Trainer


import os
import sys

# sys.path.append(f"../../{os.path.dirname(os.path.abspath(__file__))}/third_party/BigVGAN/")

import hashlib
import re
import tempfile
from importlib.resources import files

import matplotlib

matplotlib.use("Agg")

import matplotlib.pylab as plt
import numpy as np
import torch
import torchaudio
import tqdm
from pydub import AudioSegment, silence
from transformers import pipeline
from vocos import Vocos

# from f5_tts.model import CFM
from num2words import num2words
import soundfile as sf
# import gradio as gr

2025-05-29 16:25:14.883743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748535914.898402       9 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748535914.902427       9 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748535914.913184       9 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748535914.913206       9 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748535914.913208       9 computation_placer.cc:177] computation placer alr

In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)
# -----------------------------------------

target_sample_rate = 24000
n_mel_channels = 100
hop_length = 256
win_length = 1024
n_fft = 1024
mel_spec_type = "vocos"
# mel_spec_type = "bigvgan"
target_rms = 0.1
cross_fade_duration = 0.15
ode_method = "euler"
nfe_step = 32  # 16, 32
cfg_strength = 2.0
sway_sampling_coef = -1.0
speed = 1.0
fix_duration = None

# -----------------------------------------

_ref_audio_cache = {}
# load asr pipeline
asr_pipe = None

cuda


In [3]:
#UTILS_INFER
# chunk text into smaller pieces

def chunk_text(text, max_chars=135):
    """
    Splits the input text into chunks, each with a maximum number of characters.

    Args:
        text (str): The text to be split.
        max_chars (int): The maximum number of characters per chunk.

    Returns:
        List[str]: A list of text chunks.
    """
    chunks = []
    current_chunk = ""
    # Split the text into sentences based on punctuation followed by whitespace
    # sentences = re.split(r"(?<=[;:,.!?])\s+|(?<=[；：，。！？])", text)
    # sentences = re.split(r'(?<=[;:.!?"])\s+|(?<=[；：。！？"])', text)
    sentences = re.split(r'(?<=[;:.!?])\s+|(?<=[；：。！？])', text)
    for sentence in sentences:
        if len(current_chunk.encode("utf-8")) + len(sentence.encode("utf-8")) <= max_chars:
            current_chunk += sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def sentences_text(text,max_chars=135):
    """
    Splits the input text into chunks, each with a maximum number of characters.

    Args:
        text (str): The text to be split.
        max_chars (int): The maximum number of characters per chunk.

    Returns:
        List[str]: A list of text chunks.
    """
    
    # Split the text into sentences based on punctuation followed by whitespace
    sentences = re.split(r"(?<=[;:,.!?])\s+|(?<=[；：，。！？])", text)
    # sentences = re.split(r"(?<=[;:.!?])\s+|(?<=[；：。！？])", text)

    return sentences



# load vocoder
def load_vocoder(is_local=False, local_path="", device=device):
    if mel_spec_type == "vocos":
        if is_local:
            print(f"Load vocos from local path {local_path}")
            vocoder = Vocos.from_hparams(f"{local_path}/config.yaml")
            state_dict = torch.load(f"{local_path}/pytorch_model.bin", map_location="cpu")
            vocoder.load_state_dict(state_dict)
            vocoder = vocoder.eval().to(device)
        else:
            print("Download Vocos from huggingface charactr/vocos-mel-24khz")
            vocoder = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device)
    elif mel_spec_type == "bigvgan":
        try:
            # from third_party.BigVGAN import bigvgan
            import bigvgan
        except ImportError:
            print("You need to follow the README to init submodule and change the BigVGAN source code.")
        if is_local:
            """download from https://huggingface.co/nvidia/bigvgan_v2_24khz_100band_256x/tree/main"""
            vocoder = bigvgan.BigVGAN.from_pretrained(local_path, use_cuda_kernel=False)
        else:
            vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_24khz_100band_256x", use_cuda_kernel=False)

        vocoder.remove_weight_norm()
        vocoder = vocoder.eval().to(device)
    return vocoder



def initialize_asr_pipeline(device=device, dtype=None):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    global asr_pipe
    asr_pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-large-v3-turbo",
        torch_dtype=dtype,
        device=device,
    )


# load model checkpoint for inference


def load_checkpoint(model, ckpt_path, device, dtype=None, use_ema=True):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    model = model.to(dtype)

    ckpt_type = ckpt_path.split(".")[-1]
    if ckpt_type == "safetensors":
        from safetensors.torch import load_file

        checkpoint = load_file(ckpt_path)
    else:
        checkpoint = torch.load(ckpt_path, weights_only=True)

    if use_ema:
        if ckpt_type == "safetensors":
            checkpoint = {"ema_model_state_dict": checkpoint}
        checkpoint["model_state_dict"] = {
            k.replace("ema_model.", ""): v
            for k, v in checkpoint["ema_model_state_dict"].items()
            if k not in ["initted", "step"]
        }

        # patch for backward compatibility, 305e3ea
        for key in ["mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"]:
            if key in checkpoint["model_state_dict"]:
                del checkpoint["model_state_dict"][key]

        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        if ckpt_type == "safetensors":
            checkpoint = {"model_state_dict": checkpoint}
        model.load_state_dict(checkpoint["model_state_dict"])

    return model.to(device)


# load model for inference



def remove_silence_edges(audio, silence_threshold=-42):
    # Remove silence from the start
    non_silent_start_idx = silence.detect_leading_silence(audio, silence_threshold=silence_threshold)
    audio = audio[non_silent_start_idx:]

    # Remove silence from the end
    non_silent_end_duration = audio.duration_seconds
    for ms in reversed(audio):
        if ms.dBFS > silence_threshold:
            break
        non_silent_end_duration -= 0.001
    trimmed_audio = audio[: int(non_silent_end_duration * 1000)]

    return trimmed_audio



# infer process: chunk text -> infer batches [i.e. infer_batch_process()]

# remove silence from generated wav


def remove_silence_for_generated_wav(filename):
    aseg = AudioSegment.from_file(filename)
    non_silent_segs = silence.split_on_silence(
        aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=500, seek_step=10
    )
    non_silent_wave = AudioSegment.silent(duration=0)
    for non_silent_seg in non_silent_segs:
        non_silent_wave += non_silent_seg
    aseg = non_silent_wave
    aseg.export(filename, format="wav")


# save spectrogram


def save_spectrogram(spectrogram, path):
    plt.figure(figsize=(12, 4))
    plt.imshow(spectrogram, origin="lower", aspect="auto")
    plt.colorbar()
    plt.savefig(path)
    plt.close()



In [4]:
#UTILS

import os
import random
from collections import defaultdict
from importlib.resources import files

import torch
from torch.nn.utils.rnn import pad_sequence

import jieba
from pypinyin import lazy_pinyin, Style


# seed everything
def seed_everything(seed=0):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# helpers


def exists(v):
    return v is not None


def default(v, d):
    return v if exists(v) else d


def traducir_numero_a_texto(texto):
    texto_separado = re.sub(r'([A-Za-z])(\d)', r'\1 \2', texto)
    texto_separado = re.sub(r'(\d)([A-Za-z])', r'\1 \2', texto_separado)
    
    def reemplazar_numero(match):
        numero = match.group()
        return num2words(int(numero), lang='es')

    texto_traducido = re.sub(r'\b\d+\b', reemplazar_numero, texto_separado)

    return texto_traducido


# convert char to pinyin
def convert_char_to_pinyin(text_list, polyphone=True):
    final_text_list = []
    god_knows_why_en_testset_contains_zh_quote = str.maketrans(
        {"“": '"', "”": '"', "‘": "'", "’": "'"}
    )  # in case librispeech (orig no-pc) test-clean
    custom_trans = str.maketrans({";": ","})  # add custom trans here, to address oov
    for text in text_list:
        char_list = []
        text = text.translate(god_knows_why_en_testset_contains_zh_quote)
        text = text.translate(custom_trans)
        for seg in jieba.cut(text):
            seg_byte_len = len(bytes(seg, "UTF-8"))
            if seg_byte_len == len(seg):  # if pure alphabets and symbols
                if char_list and seg_byte_len > 1 and char_list[-1] not in " :'\"":
                    char_list.append(" ")
                char_list.extend(seg)
            elif polyphone and seg_byte_len == 3 * len(seg):  # if pure chinese characters
                seg = lazy_pinyin(seg, style=Style.TONE3, tone_sandhi=True)
                for c in seg:
                    if c not in "。，、；：？！《》【】—...":
                        char_list.append(" ")
                    char_list.append(c)
            else:  # if mixed chinese characters, alphabets and symbols
                for c in seg:
                    if ord(c) < 256:
                        char_list.extend(c)
                    else:
                        if c not in "。，、；：？！《》【】—...":
                            char_list.append(" ")
                            char_list.extend(lazy_pinyin(c, style=Style.TONE3, tone_sandhi=True))
                        else:  # if is zh punc
                            char_list.append(c)
        final_text_list.append(char_list)

    return final_text_list


# filter func for dirty data with many repetitions


def repetition_found(text, length=2, tolerance=10):
    pattern_count = defaultdict(int)
    for i in range(len(text) - length + 1):
        pattern = text[i : i + length]
        pattern_count[pattern] += 1
    for pattern, count in pattern_count.items():
        if count > tolerance:
            return True
    return False



In [5]:
def get_tokenizer(dataset_name, tokenizer: str = "pinyin"):
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                - "char" for char-wise tokenizer, need .txt vocab_file
                - "byte" for utf-8 tokenizer
                - "custom" if you're directly passing in a path to the vocab.txt you want to use
    vocab_size  - if use "pinyin", all available pinyin types, common alphabets (also those with accent) and symbols
                - if use "char", derived from unfiltered character & symbol counts of custom dataset
                - if use "byte", set to 256 (unicode byte range)
    """
    if tokenizer in ["pinyin", "char"]:
        tokenizer_path = os.path.join(files("f5_tts").joinpath("../../data"), f"{dataset_name}_{tokenizer}/vocab.txt")
        # tokenizer_path ="./F5TTS/vocab_pinyin.txt"
        # tokenizer_path ="/home/jupyter/F5TTS/vocab_pinyin.txt"
        with open(tokenizer_path, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)
        assert vocab_char_map[" "] == 0, "make sure space is of idx 0 in vocab.txt, cuz 0 is used for unknown char"

    elif tokenizer == "byte":
        vocab_char_map = None
        vocab_size = 256

    elif tokenizer == "custom":
        with open(dataset_name, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)

    return vocab_char_map, vocab_size


In [6]:
def preprocess_ref_audio_text(ref_audio_orig, ref_text, clip_short=False, show_info=print, device=device):
    show_info("Converting audio...")
    print("Converting audio...")
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        aseg = AudioSegment.from_file(ref_audio_orig)

        if clip_short:
            # 1. try to find long silence for clipping
            non_silent_segs = silence.split_on_silence(
                aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=1000, seek_step=10 
                # aseg, min_silence_len=2000, silence_thresh=-50, keep_silence=1000, seek_step=10 
            )
            non_silent_wave = AudioSegment.silent(duration=0)
            for non_silent_seg in non_silent_segs:
                if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                    show_info("Audio is over 15s, clipping short. (1)")
                    break
                non_silent_wave += non_silent_seg

            # 2. try to find short silence for clipping if 1. failed
            if len(non_silent_wave) > 15000:
                non_silent_segs = silence.split_on_silence(
                    aseg, min_silence_len=100, silence_thresh=-40, keep_silence=1000, seek_step=10
                )
                non_silent_wave = AudioSegment.silent(duration=0)
                for non_silent_seg in non_silent_segs:
                    if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                        show_info("Audio is over 15s, clipping short. (2)")
                        break
                    non_silent_wave += non_silent_seg

            aseg = non_silent_wave

            # 3. if no proper silence found for clipping
            if len(aseg) > 15000:
                aseg = aseg[:15000]
                show_info("Audio is over 15s, clipping short. (3)")

        aseg = remove_silence_edges(aseg) + AudioSegment.silent(duration=50)
        aseg.export(f.name, format="wav")
        ref_audio = f.name

    # Compute a hash of the reference audio file
    with open(ref_audio, "rb") as audio_file:
        audio_data = audio_file.read()
        audio_hash = hashlib.md5(audio_data).hexdigest()

    global _ref_audio_cache
    if audio_hash in _ref_audio_cache:
        # Use cached reference text
        show_info("Using cached reference text...")
        ref_text = _ref_audio_cache[audio_hash]
    else:
        print("ref_text ",len(ref_text),ref_text)
        if not ref_text.strip():
        # if len(ref_text)==0:
        # if ref_text =="":
            
            global asr_pipe
            if asr_pipe is None:
                initialize_asr_pipeline(device=device)
            show_info("No reference text provided, transcribing reference audio...")
            ref_text = asr_pipe(
                ref_audio,
                chunk_length_s=30,
                batch_size=128,
                generate_kwargs={"task": "transcribe"},
                return_timestamps=False,
            )["text"].strip()
            
            show_info("Finished transcription")
        else:
            show_info("Using custom reference text...")
        # Cache the transcribed text
        _ref_audio_cache[audio_hash] = ref_text
    print("ref_text: ",ref_text)
    # Ensure ref_text ends with a proper sentence-ending punctuation
    if not ref_text.endswith(". ") and not ref_text.endswith("。"):
        if ref_text.endswith("."):
            ref_text += " "
        else:
            ref_text += ". "

    return ref_audio, ref_text


In [7]:
def load_model(
    model_cls,
    model_cfg,
    ckpt_path,
    mel_spec_type=mel_spec_type,
    vocab_file="",
    ode_method=ode_method,
    use_ema=True,
    device=device,
):
    if vocab_file == "":
        # vocab_file = str(files("f5_tts").joinpath("infer/examples/vocab.txt"))
        vocab_file = "./F5TTS/vocab.txt"
    tokenizer = "custom"
    # tokenizer = "pinyin"
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                    - "char" for char-wise tokenizer, need .txt vocab_file
                    - "byte" for utf-8 tokenizer
                    - "custom" if you're directly passing in a path to the vocab.txt you want to use
    """

    print("\nvocab : ", vocab_file)
    print("tokenizer : ", tokenizer)
    print("model : ", ckpt_path, "\n")

    vocab_char_map, vocab_size = get_tokenizer(vocab_file, tokenizer)
    model = CFM(
        transformer=model_cls(**model_cfg, text_num_embeds=vocab_size, mel_dim=n_mel_channels),
        mel_spec_kwargs=dict(
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mel_channels=n_mel_channels,
            target_sample_rate=target_sample_rate,
            mel_spec_type=mel_spec_type,
        ),
        odeint_kwargs=dict(
            method=ode_method,
        ),
        vocab_char_map=vocab_char_map,
    ).to(device)

    dtype = torch.float32 if mel_spec_type == "bigvgan" else None
    model = load_checkpoint(model, ckpt_path, device, dtype=dtype, use_ema=use_ema)

    return model



In [8]:
ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozRealSeriaMarcos3.wav"))+"/VozRealSeriaMarcos3.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozPresentadorMarcos3.wav"))+"/VozPresentadorMarcos3.wav"#"./F5TTS/FL.wav", #Ruta al audio


ref_text_input=""
# ref_text_input='Laura medía un metro setenta, tenía los pies palmeados de nacimiento y marcas de nacimiento iguales en ambos muslos. Una tenía la forma de su padre y la otra la de su madre, o eso decía ella. A mí me parecían salpicaduras negras más o menos iguales, la izquierda ligeramente más grande, más dentada que la otra, ambas moteadas por manchas de marrón oscuro. Un solo pelo salía largo de la más suave. Me las enseñó tres semanas después de conocernos en un banco mojado de un parque a las tres de la madrugada. Sus pies palmeados aparecieron primero, pero no le preocupaban mucho. Dijo que no veía el alboroto. No desde el instituto. Sus curvas y su baja estatura la convertían en una pésima nadadora y las otras chicas habían hecho un deporte de señalar la ironía, entre otras cosas más mezquinas. Laura era irónica, en muchos aspectos más que eso. La marca de su padre era la más dolorosa. Se le llenaban los ojos de lágrimas mientras trazaba los contornos del borde más afilado y explicaba el significado de su extraña geometría. Pero era difícil seguir después de la parte del bastón de madera. Hablaba a trompicones y cada tres o cuatro palabras sonaban a árabe; y resultó que era árabe. El árabe es un idioma impresionante. Hasta las indicaciones para ir al baño suenan poéticas en árabe. Al menos para mí.   Nunca le pregunté qué significaba, no me preguntes por qué, y traducir sus palabras ahora, después de lo que pasó -después de lo que hizo- es lo más alejado de cualquier cosa que pueda imaginarme haciendo por elección propia.   Lo mismo ocurrió con la marca por parte de madre, pero fue el italiano el idioma al que se dirigió entonces.  Estaba demasiado hipnotizado para decir nada.  Sólo seguía sus expresiones e inflexiones lo mejor que podía. Cuando su lengua cambió, sentí un dolor diferente, más intenso, por lo que pude ver, en la zona donde crecía el pelo largo.  Era casi imposible no abrazarla cuando se estremecía. Y entonces el propio pelo me hizo sonreír, lo suficiente como para mostrar lo compleja que era aquella relación.  Nunca había conocido a una chica con metáforas naturales en las piernas. No es que ella lo viera así, claro.   No podía ser más entrañable, y su trauma hizo que se me encendieran las entrañas. No creo que importara mucho que yo no lo siguiera todo.  Todo giraba en torno a ella. Yo era su seguridad más bien, que era como había sido desde el principio.  Desde la noche en que volvía tarde a casa y la encontré agarrada al otro lado de la barrera. La del puente alto sobre el río.'


remove_silence=False #El modelo tiende a producir silencios, especialmente en audios más largos. Podemos eliminar manualmente los silencios si es necesario. Ten en cuenta que esta es una característica experimental y puede producir resultados extraños. Esto también aumentará el tiempo de generación.
cross_fade_duration_slider=0.15 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
# cross_fade_duration_slider=1.0 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
speed_slider=2.0#Ajusta la velocidad del audio. Entre 0.3 y 2.0


In [9]:
# def infer(ref_audio_orig, ref_text, gen_text, remove_silence, cross_fade_duration=0.15, speed=1):
# ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=False)
ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=True)
# print(ref_text)


Converting audio...
Converting audio...
ref_text  0 


Device set to use cuda
/usr/local/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


No reference text provided, transcribing reference audio...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Finished transcription
ref_text:  Al pasar de 10.000 a 5.000 kilómetros, las lecturas de radiación se dispararon y los niveles de microgravedad comenzaron a fluctuar sin motivo aparente. Ante ellos se erguía la silueta de aquello que el espacio había ocultado durante milenios.


In [20]:
# Noches de san damian
# Cap 1
gen_text_input='Mi nombre es Heusebio Mendoza, y durante treinta años he recorrido los caminos polvorientos de esta tierra llevando mercancías de pueblo en pueblo. He visto muchas cosas en mis viajes: tormentas que parecían el fin del mundo, bandidos que aparecían como sombras en la noche, y pueblos tan prósperos como miserables. Pero nunca, en todos mis años sobre el pescante de mi carreta, había experimentado algo como lo que me sucedió en San Damián.'

In [29]:
# Noches de san damian
# Cap 2
gen_text_input='Esa primera noche cenamos en el comedor común de la posada: un guiso de cordero con papas y un vino tinto local que calentaba el alma. Compartí la mesa con otros huéspedes: un comerciante de herramientas, un cura joven que viajaba a su nueva parroquia, y un arriero veterano llamado Jacinto que conocía estos caminos mejor que nadie. "¿Van a quedarse mañana?" preguntó doña Remedios mientras servía el postre. "¿Por qué habríamos de hacerlo?" respondió el comerciante. La mujer intercambió una mirada preocupada con su marido, don Aurelio, quien había permanecido callado durante toda la cena. "Es que mañana es día de los difuntos," explicó finalmente. "Y aquí en San Damián... bueno, es mejor no viajar esa noche." Jacinto, el arriero veterano, dejó escapar una risa seca. "Supersticiones de pueblo, doña Remedios. Los muertos están muertos, y los vivos tenemos que trabajar.". Pero yo noté algo en los ojos de la posadera, una inquietud que no podía disimular. Había visto esa misma expresión en otros lugares, en otros pueblos donde las tradiciones antiguas aún pesaban más que la razón moderna. Esa noche, mientras me preparaba para dormir en mi habitación del segundo piso, escuché un sonido extraño. Era como un murmullo, voces que conversaban en voz baja, pero cuando me asomé a la ventana que daba al patio, no vi a nadie. Las mulas estaban inquietas, moviéndose nerviosamente y relinchando de vez en cuando. "Será el viento," me dije, pero sabía que no había viento esa noche. El aire estaba completamente quieto, como si el pueblo entero hubiera contenido la respiración. Me dormí con dificultad, y mis sueños estuvieron poblados de figuras borrosas que caminaban entre las sombras.'

In [37]:
# Noches de san damian
# Cap 3
gen_text_input='Desperté antes del amanecer con la intención de partir temprano, pero al mirar por la ventana descubrí que una niebla espesa había cubierto todo el pueblo. Era tan densa que apenas podía ver el patio de la posada. Sabía que viajar en esas condiciones sería peligroso; un paso en falso podría llevar mi carreta por un barranco. Bajé al comedor y encontré a los demás huéspedes en la misma situación. La niebla había llegado como una manta gris que lo cubría todo. "Esto es inusual," comentó el padre Sebastián, el cura joven, mientras miraba por la ventana. "En mi pueblo natal nunca he visto una niebla tan espesa." Don Aurelio apareció con una bandeja de café y pan tostado. Su rostro lucía más grave que la noche anterior. "La niebla se levantará al mediodía," nos dijo. "Siempre pasa lo mismo el día de los difuntos." "¿Siempre?" preguntó el comerciante. "¿Qué quiere decir con siempre?". Don Aurelio se quedó callado, pero su esposa, doña Remedios, se acercó limpiándose las manos en el delantal. "Cada año, el primero de noviembre, baja la niebla desde las montañas," explicó en voz baja. "Y cada año, quienes están en el pueblo esa noche... bueno, escuchan cosas." "¿Qué tipo de cosas?" insistió el padre Sebastián. "Voces. Pasos. Puertas que se abren y se cierran solas." La mujer se persignó. "Son las almas en pena que regresan a visitar a sus familiares." Jacinto bufó desde su rincón. "Tonterías de gente ignorante." Pero yo recordé los murmullos de la noche anterior, y un escalofrío me recorrió la espalda. La mañana transcurrió lentamente. La niebla no se disipaba, y de hecho parecía volverse más densa. Desde el comedor podíamos escuchar sonidos extraños: pasos que caminaban por las calles empedradas, voces que conversaban en idiomas que no podíamos entender, y ocasionalmente, el sonido de carruajes que pasaban como fantasmas. "¿Hay otros viajeros en el pueblo?" pregunté a don Aurelio. "No," respondió secamente. "Ustedes son los únicos huéspedes." El comerciante se acercó a la ventana y pegó el rostro al vidrio. "Creo que veo figuras moviéndose allá afuera." Todos nos acercamos, pero la niebla era tan espesa que era imposible distinguir formas definidas. Sin embargo, había algo ahí, sombras que se movían con propósito, como si caminaran hacia algún destino conocido.'

In [ ]:
# Noches de san damian
# Cap 4
gen_text_input='Al caer la tarde, la niebla comenzó a aclararse ligeramente, lo suficiente para que pudiéramos ver las calles inmediatas. Decidí salir a revisar mi carreta y asegurarme de que las mulas estuvieran bien. El padre Sebastián me acompañó, curioso por explorar el pueblo. Las calles estaban completamente desiertas. Nuestros pasos resonaban sobre las piedras mojadas por la humedad, creando ecos extraños que parecían multiplicarse. Las casas tenían las ventanas cerradas y las puertas bien aseguradas. "Es como si el pueblo entero hubiera desaparecido," murmuró el padre Sebastián. Llegamos al pequeño corral donde había dejado mi carreta. Las mulas estaban nerviosas, con los ojos muy abiertos y las orejas alerta. Una de ellas, la más vieja y normalmente la más tranquila, temblaba visiblemente. "Algo las tiene asustadas," observé, acariciando el cuello de la mula. Fue entonces cuando lo escuchamos: un sonido distante, como el de muchos pies caminando al unísono. Se acercaba lentamente, viniendo desde la dirección de la iglesia. "¿Es una procesión?" preguntó el padre Sebastián. Nos quedamos quietos, escuchando. El sonido se hacía más claro: pasos, muchos pasos, y ocasionalmente el murmullo de voces que rezaban. Pero había algo extraño en esas voces, algo que me erizaba la piel. "Deberíamos regresar a la posada," sugerí. Pero el padre Sebastián, movido por la curiosidad clerical, quería ver de qué se trataba. "Si es una procesión religiosa, debería participar." Comenzamos a caminar hacia el sonido, que ahora se había convertido en un coro de voces que entonaban lo que parecía ser un cántico fúnebre. La niebla se arremolinaba a nuestro alrededor, creando formas fantasmales que aparecían y desaparecían. Al doblar una esquina, los vimos. Una larga fila de figuras caminaba lentamente por la calle principal, dirigiéndose hacia el cementerio en las afueras del pueblo. Pero estas figuras no eran como las personas normales. Sus ropas parecían de otra época, y sus movimientos tenían una cualidad etérea, como si flotaran más que caminar. El padre Sebastián se detuvo bruscamente, y yo sentí que se me helaba la sangre. Algunas de las figuras llevaban velas que ardían con una llama azul pálida, otras portaban flores que parecían marchitas y negras. Sus rostros... sus rostros eran borrosos, como si estuvieran cubiertos por un velo de agua. "Dios mío," susurró el padre Sebastián, y comenzó a rezar en voz baja. La procesión pasó a nuestro lado sin notarnos, o al menos eso pareció. Pero cuando la última figura se acercaba, se detuvo y giró la cabeza hacia nosotros. No podía ver sus rasgos claramente, pero sentí que me observaba con una intensidad que me atravesaba el alma. Entonces, con una voz que sonaba como el viento entre las hojas secas, habló: "Los vivos no deben caminar con los muertos en esta noche."'

In [ ]:
# Noches de san damian
# Cap 5
gen_text_input='Regresamos a la posada corriendo, con el corazón latiéndonos como tambores de guerra. Encontramos a los demás huéspedes reunidos en el comedor, donde doña Remedios había encendido todas las velas disponibles. "¿Vieron la procesión?" preguntó don Aurelio sin levantar la vista del suelo. No pudimos hacer otra cosa que asentir. El comerciante, quien había sido el más escéptico, ahora temblaba visiblemente. "¿Qué era eso?" preguntó con voz quebrada. Doña Remedios se sentó pesadamente en una silla y comenzó a contarnos la historia que había permanecido oculta durante nuestra estancia. "Hace cincuenta años, San Damián era un pueblo próspero. La mina de plata en las montañas daba trabajo a todos, y las familias vivían bien. Pero una noche de noviembre, la noche de los difuntos, hubo un derrumbe terrible en la mina.". Su voz se volvió más grave mientras continuaba. "Murieron setenta hombres de una sola vez. Padres, hijos, hermanos... familias enteras perdieron a sus sostenedores. Pero eso no fue lo peor.". "¿Qué más pasó?" preguntó el padre Sebastián. "Los cuerpos nunca fueron recuperados. La mina se selló para siempre, y esas almas quedaron atrapadas ahí dentro. Desde entonces, cada año en la noche de los difuntos, regresan al pueblo. Caminan desde la mina hasta el cementerio, buscando el descanso que nunca han encontrado.". Jacinto, el arriero veterano, había permanecido callado durante toda la explicación, pero ahora habló con voz temblorosa: "Yo... yo conocía esa historia. Mi abuelo trabajaba en esa mina. Murió en el derrumbe.". Se hizo un silencio sepulcral en el comedor. Las velas parpadeaban, creando sombras danzantes en las paredes. "¿Por qué no nos dijeron antes?" preguntó el comerciante. "Porque la mayoría de los viajeros no se quedan en esa fecha," respondió don Aurelio. "Y los que se han quedado... bueno, algunos prefieren olvidar lo que vieron.". Afuera, el viento había comenzado a soplar, llevándose gradualmente la niebla. Pero con él llegaron nuevos sonidos: lamentos que parecían venir de las profundidades de la tierra, puertas que se abrían y cerraban en casas vacías, y ocasionalmente, el sonido de picos y palas como si alguien siguiera trabajando en la mina abandonada. '

In [124]:
# Noches de san damian
# Cap 6
gen_text_input='Las horas pasaron con una lentitud agobiante. Ninguno de nosotros podía dormir, así que permanecimos en el comedor, bebiendo café y aguardiente, manteniéndonos despiertos con historias y oraciones. El padre Sebastián había sacado su breviario y rezaba en voz baja, mientras que el comerciante no paraba de caminar de un lado a otro. Jacinto se había sumido en un silencio profundo, perdido en recuerdos que probablemente hubiera preferido mantener enterrados. Cerca de la medianoche, los sonidos se intensificaron. Ahora podíamos escuchar claramente voces que llamaban nombres, como si los muertos estuvieran buscando a sus familiares vivos. Ocasionalmente, alguien golpeaba las puertas de las casas del pueblo, pero nadie respondía. "¿No hay nadie más en el pueblo?" pregunté a doña Remedios. "Claro que sí," respondió. "Pero en esta noche, nadie sale de su casa. Todas las puertas están bendecidas con agua santa, y las ventanas protegidas con cruces y ramas de ruda.". Como para confirmar sus palabras, escuchamos golpes en la puerta principal de la posada. No eran golpes fuertes o amenazantes, sino más bien como el toque suave de alguien que pide permiso para entrar. Don Aurelio se dirigió hacia la puerta con una vela en la mano, pero su esposa le gritó: "¡No abras! ¡Nunca abras en esta noche!". Los golpes continuaron durante varios minutos, acompañados de una voz que parecía familiar pero que no podíamos identificar. Llamaba por el nombre de don Aurelio, pidiendo refugio del frío. "Es mi hermano Miguel," susurró don Aurelio con los ojos llenos de lágrimas. "Murió en el derrumbe.". Doña Remedios se acercó a su marido y le tomó las manos. "No es tu hermano, Aurelio. Es solo su recuerdo, su eco. El verdadero Miguel está en paz.". Pero los golpes continuaron, y la voz se volvió más insistente, más desesperada. Finalmente, cuando el reloj de la iglesia marcó la una de la mañana, todo se quedó en silencio.'

In [141]:
# Noches de san damian
# Cap 7
gen_text_input='Las primeras luces del alba trajeron consigo un silencio diferente, un silencio de paz en lugar de expectación. La niebla se había disipado completamente, y por las ventanas entraba la luz dorada del sol. Nos habíamos quedado dormidos en nuestras sillas, agotados por la vigilia. Desperté cuando los rayos del sol tocaron mi rostro, y por un momento, pensé que todo había sido un sueño extraño. Pero las caras de mis compañeros de viaje mostraban la misma expresión de quien ha visto algo que no puede explicar fácilmente. Salimos al patio de la posada en silencio, como si temiéramos romper la tranquilidad que había regresado al pueblo. Mis mulas estaban calmadas ahora, pastando tranquilamente en su corral. La carreta seguía donde la había dejado, intacta. Todo parecía normal, como si la noche anterior hubiera sido producto de nuestra imaginación colectiva. Pero cuando fui a revisar la carga, encontré algo que me heló la sangre: en la lona que cubría las mercancías había una huella de mano, impresa en lo que parecía ser tierra húmeda de cementerio. El padre Sebastián se acercó y observó la marca con atención. "No es tierra común," murmuró. "Huele a... a tiempo.". Preparamos nuestros equipajes en silencio. El comerciante había decidido acompañarme hasta Valparaíso, alegando que tenía asuntos urgentes que atender. Jacinto, por su parte, había cambiado su ruta y viajaría hacia el norte, hacia tierras que no conociera tan íntimamente. Doña Remedios nos preparó un desayuno abundante, pero nadie tenía mucho apetito. Antes de partir, don Aurelio me llevó aparte. "Don Eusebio," me dijo en voz baja, "si alguien le pregunta sobre lo que vio anoche, es mejor que diga que no vio nada. La gente de otros pueblos no entendería.". Le aseguré que mantendría el secreto, aunque sabía que esa experiencia me acompañaría por el resto de mis días.'

In [ ]:
# Noches de san damian
# Cap 8
gen_text_input='Partimos de San Damián cuando el sol ya estaba alto en el cielo. El pueblo lucía completamente normal: las personas realizaban sus actividades cotidianas, los niños jugaban en las calles, y los comerciantes abrían sus tiendas como cualquier otro día. Era como si la noche anterior hubiera pertenecido a un mundo diferente, a una realidad paralela que solo existía en la oscuridad. Durante el viaje hacia Valparaíso, el comerciante y yo hablamos poco sobre lo sucedido. Había una comprensión tácita entre nosotros de que algunas experiencias deben procesarse en silencio antes de poder ser compartidas. Completé mi viaje sin más incidentes y entregué mi carga en los almacenes del puerto. Los comerciantes de Valparaíso pagaron bien por las telas y especias, pero el dinero parecía insignificante comparado con lo que había experimentado. Pasaron los meses, y poco a poco comencé a dudar de mis propios recuerdos. ¿Había sido real la procesión de los muertos? ¿O había sido una alucinación colectiva causada por el cansancio y las historias de fantasmas? Pero cada vez que pasaba cerca de San Damián en mis viajes posteriores, tomaba rutas alternativas. No por miedo, sino por respeto. Había lugares en este mundo donde los vivos y los muertos compartían el mismo espacio, y esa noche había aprendido que no todos los misterios necesitan ser resueltos.'

In [170]:
# Noches de san damian
# Cap 9
gen_text_input='Han pasado diez años desde aquella noche en San Damián, y ahora soy un hombre mayor que ha comenzado a pensar en colgar las riendas para siempre. Mis cabellos se han vuelto grises, y mis huesos ya no toleran tan bien los caminos rocosos. Pero aún conservo la lona de mi carreta con la huella de esa mano misteriosa. Con los años, la marca no se ha desvanecido; al contrario, parece haberse vuelto más nítida, como si el tiempo la hubiera grabado más profundamente en la tela. Hace poco, un joven arriero me preguntó sobre esa marca mientras cargábamos mercancías en Valparaíso. Le dije que era una vieja mancha de trabajo, pero él me miró con ojos conocedores. "Mi abuelo tenía marcas similares en sus equipos," me dijo. "Decía que eran recuerdos de lugares donde había visto cosas que no podía explicar.". Le sonreí y no dije nada más. Pero esa noche, mientras ordenaba mis pertenencias en la habitación de la posada, encontré una carta que había llegado esa mañana. El remitente era el padre Sebastián, aquel joven cura que había compartido conmigo la experiencia en San Damián. La carta decía: "Estimado don Eusebio, Espero que esta carta lo encuentre con buena salud. Le escribo desde mi parroquia en las montañas del norte, donde he servido durante estos años. Quería contarle algo que creo le interesará. Hace poco, investigando los archivos de la iglesia, encontré documentos sobre la tragedia de la mina de San Damián. Los registros confirman que efectivamente murieron setenta hombres en el derrumbe de 1797, exactamente cincuenta años antes de nuestra visita. Pero hay algo más: encontré testimonios de viajeros que datan de diferentes años, todos describiendo experiencias similares a la nuestra en la noche de los difuntos. Algunos de estos testimonios son de décadas anteriores a nuestro encuentro. No sé qué pensar de todo esto, pero creo que usted, como yo, ha sido testigo de algo que trasciende nuestra comprensión habitual del mundo. Que Dios lo bendiga en sus caminos. Padre Sebastián Morales" Doblé la carta cuidadosamente y la guardé junto con otros documentos importantes. Afuera, la noche había caído sobre Valparaíso, y el sonido de las olas contra el puerto creaba una melodía constante. Miré por la ventana hacia las montañas distantes, donde sabía que San Damián seguía existiendo, con sus calles empedradas y su iglesia colonial. Y me pregunté si en este momento, en algún lugar de esas montañas, otro viajero estaría experimentando lo mismo que yo viví hace diez años. Porque hay verdades en este mundo que no cambian con el tiempo, verdades que persisten más allá de la comprensión humana. Y San Damián, con sus muertos que caminan y sus secretos enterrados, es una de ellas. Mañana comenzaré mi último viaje como carretero. He decidido que es hora de establecerme en un lugar y dejar que otros más jóvenes recorran estos caminos. Pero antes de hacerlo, haré una parada final. Tengo que regresar a San Damián. No para quedarse en la noche de los difuntos, sino para devolver algo que he llevado conmigo durante todos estos años: esa lona con la huella de la mano misteriosa. Tengo la sensación de que pertenece allí, de que es un pedazo de ese mundo entre mundos que debe regresar a su lugar de origen. Y tal vez, cuando la entregue, mis propios fantasmas finalmente puedan descansar en paz.'

In [10]:
# Noches de san damian
# Presentación
gen_text_input='Hay caminos que todos los arrieros conocen, y hay pueblos donde ningún viajero sensato se queda después del anochecer. Eusebio Mendoza, creía que después de treinta años recorriendo los mismos senderos polvorientos había visto de todo... hasta que una vez una niebla extraña lo obligó a pasar el día de muertos en San Damián. Lo que vio esa noche cambió para siempre su manera de entender el mundo. Porque hay lugares donde los muertos no descansan, donde el tiempo se detiene una vez al año, y donde una simple huella en la lona de una carreta puede perseguirte durante décadas. Esta es la historia de Eusebio Mendoza y su encuentro con lo imposible, una historia que ha guardado en silencio durante años... hasta ahora.'

In [10]:
gen_text_input='Bienvenidos viajeros, soy El Carretero, y hoy los llevaré por senderos donde la realidad se desvanece y lo imposible cobra vida. Prepárense para adentrarse en historias que desafían la razón, donde las sombras susurran secretos ancestrales y fuerzas cósmicas despiertan desde las profundidades del tiempo. Suban a mi carreta... el viaje hacia lo desconocido está por comenzar.'

In [21]:
# ruta="./F5TTS/Presentaciones/"
ruta="./F5TTS/NochesSanDamian/"

# ema_model = F5TTS_ema_model

# if not gen_text_input.startswith(" "):
# 	gen_text_input = " " + gen_text_input
# if not gen_text_input.endswith(". "):
# 	gen_text_input += ". "

# gen_text_input = gen_text_input.lower()
# gen_text_input = traducir_numero_a_texto(gen_text_input)

# print (gen_text_input)
# audio, sr = torchaudio.load(ref_audio)
# max_chars = int(len(ref_text.encode("utf-8")) / (audio.shape[-1] / sr) * (25 - audio.shape[-1] / sr))
# print(max_chars)
# gen_text_batches = chunk_text(gen_text_input, max_chars=max_chars)
original_batches = chunk_text(gen_text_input, max_chars=80)
# gen_text_batches = sentences_text(gen_text_input)
for batch in original_batches:
	# print(f"'{batch}',")
	print(f"'{batch}")


'Mi nombre es Heusebio Mendoza, y durante treinta años he recorrido los caminos polvorientos de esta tierra llevando mercancías de pueblo en pueblo.
'He visto muchas cosas en mis viajes:
'tormentas que parecían el fin del mundo, bandidos que aparecían como sombras en la noche, y pueblos tan prósperos como miserables.
'Pero nunca, en todos mis años sobre el pescante de mi carreta, había experimentado algo como lo que me sucedió en San Damián.


In [12]:
def procesatexto(string):
	r=traducir_numero_a_texto(string).lower()
	r=r.replace(";",",")
	r=r.replace("¡","")
	r=r.replace("!","")
	return r

## Infiriendo Step by Step

In [13]:
# _ref_audio_cache = {}
# load asr pipeline
# asr_pipe = None
vocoder = load_vocoder()
# load models
F5TTS_model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
F5TTS_ema_model = load_model(
    DiT, F5TTS_model_cfg, "./F5TTS/model_1250000.safetensors"
    # DiT, F5TTS_model_cfg, "./F5TTS/model_1200000.safetensors"
)
model_obj = F5TTS_ema_model
audio, sr = torchaudio.load(ref_audio)
try:
	os.mkdir(ruta)
	print("RUTA CREADA",ruta)
except:
	 print("La ruta ya existe",ruta)

progress=tqdm

if audio.shape[0] > 1:
	audio = torch.mean(audio, dim=0, keepdim=True)

rms = torch.sqrt(torch.mean(torch.square(audio)))
if rms < target_rms:
	audio = audio * target_rms / rms
if sr != target_sample_rate:
	resampler = torchaudio.transforms.Resample(sr, target_sample_rate)
	audio = resampler(audio)
audio = audio.to(device)

generated_waves = []
spectrograms = []

if len(ref_text[-1].encode("utf-8")) == 1:
	ref_text = ref_text + " "

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  ./F5TTS/vocab.txt
tokenizer :  custom
model :  ./F5TTS/model_1250000.safetensors 

La ruta ya existe ./F5TTS/NochesSanDamian/


In [180]:
# seed_everything(1258)

In [22]:
i=20
# j=1
# for gen_text in progress.tqdm(gen_text_batches):
for gen_text in progress.tqdm(original_batches):
	# Prepare the text
	gen_text=procesatexto(gen_text)
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for j in range(1):
		print(f"Iniciando Inferencia {i:04d}.{j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]
			generated_mel_spec = generated.permute(0, 2, 1)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			# with open(f'{ruta}{i:04d}.{j}.wav', 'w') as f:
			sf.write(f'{ruta}{i:04d}.{j}.wav', generated_wave,target_sample_rate)
			
	i+=1
	# i+=10
	

  0%|                                                                                                                                                                   | 0/4 [00:00<?, ?it/s]

Iniciando Inferencia 0020.0


 25%|██████████████████████████████████████▊                                                                                                                    | 1/4 [00:05<00:17,  5.87s/it]

Iniciando Inferencia 0021.0


 50%|█████████████████████████████████████████████████████████████████████████████▌                                                                             | 2/4 [00:10<00:09,  4.98s/it]

Iniciando Inferencia 0022.0


 75%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                      | 3/4 [00:15<00:05,  5.25s/it]

Iniciando Inferencia 0023.0


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:21<00:00,  5.34s/it]


In [31]:
# gen_text_batches=['Mi nombre es Heusebio Mendoza, y durante treinta años he recorrido los caminos polvorientos de esta tierra llevando mercancías de pueblo en pueblo.'] 
gen_text_batches=['creía que después de treinta años había visto todo...'] 
i=13
j=0
intentos=3

# i+=5
for _, gen_text in enumerate(progress.tqdm(gen_text_batches)):
# Prepare the text
	gen_text=procesatexto(gen_text)
	print (gen_text)
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for _ in range(intentos):
		print(f"Iniciando Inferencia {j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]

			#########################################################33
			if ref_audio_len >= generated.shape[1]:
				print(f"[WARN] ref_audio_len ({ref_audio_len}) >= generated length ({generated.shape[1]}), skipping slice.")
				generated_trimmed = generated  # o considera usar generated[:, -1:, :] como fallback
			else:
				generated_trimmed = generated[:, ref_audio_len:, :]

			generated_mel_spec = generated_trimmed.permute(0, 2, 1)

			###########################################################
			# print("generated shape:", generated.shape)
			# print("generated_mel_spec shape:", generated_mel_spec.shape)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			sf.write(f'{ruta}{i:04d}.{j}.wav', generated_wave,target_sample_rate)
		j+=1
	i+=1
	j=0

  0%|                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]

creía que después de treinta años había visto todo...
Iniciando Inferencia 0
[WARN] ref_audio_len (1359) >= generated length (311), skipping slice.
Iniciando Inferencia 1
[WARN] ref_audio_len (1359) >= generated length (311), skipping slice.
Iniciando Inferencia 2


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:13<00:00, 13.32s/it]

[WARN] ref_audio_len (1359) >= generated length (311), skipping slice.


In [24]:
# Combine all generated waves with cross-fading
if cross_fade_duration <= 0:
	# Simply concatenate
	final_wave = np.concatenate(generated_waves)
else:
	final_wave = generated_waves[0]
	for i in range(1, len(generated_waves)):
		prev_wave = final_wave
		next_wave = generated_waves[i]

		# Calculate cross-fade samples, ensuring it does not exceed wave lengths
		cross_fade_samples = int(cross_fade_duration * target_sample_rate)
		cross_fade_samples = min(cross_fade_samples, len(prev_wave), len(next_wave))

		if cross_fade_samples <= 0:
			# No overlap possible, concatenate
			final_wave = np.concatenate([prev_wave, next_wave])
			continue

		# Overlapping parts
		prev_overlap = prev_wave[-cross_fade_samples:]
		next_overlap = next_wave[:cross_fade_samples]

		# Fade out and fade in
		fade_out = np.linspace(1, 0, cross_fade_samples)
		fade_in = np.linspace(0, 1, cross_fade_samples)

		# Cross-faded overlap
		cross_faded_overlap = prev_overlap * fade_out + next_overlap * fade_in

		# Combine
		new_wave = np.concatenate(
			[prev_wave[:-cross_fade_samples], cross_faded_overlap, next_wave[cross_fade_samples:]]
		)

		final_wave = new_wave

# Create a combined spectrogram
combined_spectrogram = np.concatenate(spectrograms, axis=1)
final_sample_rate=target_sample_rate
# return final_wave, target_sample_rate, combined_spectrogram

In [ ]:
# if remove_silence:
with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
	sf.write(f.name, final_wave, final_sample_rate)
	remove_silence_for_generated_wav(f.name)
	final_wave, _ = torchaudio.load(f.name)
final_wave = final_wave.squeeze().cpu().numpy()

In [ ]:
sf.write(ruta+'FuenteJuventudComplete.wav', final_wave,final_sample_rate)